In [ ]:
import json
import gc
from pathlib import Path
from typing import List

import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, LlavaForConditionalGeneration

In [ ]:
MODEL_ID = "llava-hf/llava-1.5-7b-hf"

AMBER_IMG_PATH = "../data/amber/image"
QUERY_GENERATIVE_PATH = "../evaluation/AMBER/data/query/query_generative.json"
OUTPUT_PATH = "../results/infer/amber/llava15/opera.json"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 256

# OPERA parameters
NUM_ATTN_CANDIDATES = 5
SCALE_FACTOR = 50.0
THRESHOLD = 15.0
PENALTY_WEIGHTS = 1.0

MIN_ID = 1
MAX_ID = 1004

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",  # needed for output_attentions=True
).eval()

print("Processor and model loaded.")

In [ ]:
def batch_list(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def clean_output(text: str) -> str:
    text = text.strip()

    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()

    return text.strip()


def save_json_atomic(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path.with_suffix(path.suffix + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    tmp_path.replace(path)

In [ ]:
def get_llava_image_span_from_inputs(model, processor, input_ids, pixel_values):
    """
    Estimate expanded image-token span for HF LLaVA.

    Original prompt has one <image> token.
    HF LLaVA internally replaces it with visual patch features.
    """

    image_token_id = getattr(model.config, "image_token_index", None)

    if image_token_id is None:
        image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")

    image_positions = (input_ids[0] == image_token_id).nonzero(as_tuple=False)

    if len(image_positions) == 0:
        raise ValueError("Could not find <image> token in input_ids.")

    image_token_pos = int(image_positions[0].item())

    with torch.inference_mode():
        vision_outputs = model.vision_tower(
            pixel_values,
            output_hidden_states=True,
            return_dict=True,
        )

        selected = vision_outputs.hidden_states[model.config.vision_feature_layer]

        if getattr(model.config, "vision_feature_select_strategy", "default") == "default":
            selected = selected[:, 1:]

        num_image_tokens = selected.shape[1]

    image_start = image_token_pos
    image_end = image_start + num_image_tokens

    # Expanded prompt length after replacing one <image> token with visual tokens.
    response_start = input_ids.shape[1] - 1 + num_image_tokens

    return {
        "image_start": image_start,
        "image_end": image_end,
        "response_start": response_start,
    }

In [ ]:
def compute_opera_overtrust_penalty(
    attentions,
    key_position,
    scale_factor: float = 50.0,
    threshold: float = 15.0,
    penalty_weights: float = 1.0,
):
    """
    OPERA-style over-trust penalty.

    Looks at the current token's self-attention to previous response tokens.
    If attention over-focuses on previous summary tokens, apply penalty.
    """

    if attentions is None:
        raise ValueError(
            "attentions is None. Reload model with attn_implementation='eager'."
        )

    # Last decoder layer attention: [B, H, Q, K]
    last_layer_attn = attentions[-1]

    # Current token attention over keys: [B, K]
    attn = last_layer_attn[:, :, -1, :].mean(dim=1)

    response_start = key_position["response_start"]
    seq_len = attn.shape[-1]

    # No generated response token yet.
    if response_start >= seq_len - 1:
        return torch.zeros(attn.shape[0], device=attn.device, dtype=attn.dtype)

    response_attn = attn[:, response_start:seq_len - 1]

    if response_attn.numel() == 0:
        return torch.zeros(attn.shape[0], device=attn.device, dtype=attn.dtype)

    max_summary_attn = response_attn.max(dim=-1).values

    overtrust = torch.relu(max_summary_attn * scale_factor - threshold)

    penalty = penalty_weights * overtrust / scale_factor

    return penalty

In [ ]:
@torch.inference_mode()
def opera_generate_one_hf(
    model,
    processor,
    image,
    prompt: str,
    max_new_tokens: int = 256,
    num_attn_candidates: int = 5,
    scale_factor: float = 50.0,
    threshold: float = 15.0,
    penalty_weights: float = 1.0,
):
    model.eval()

    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype

    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"

    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
        padding=True,
    )

    processor.tokenizer.padding_side = old_padding_side

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    pixel_values = inputs["pixel_values"].to(device=device, dtype=dtype)

    key_position = get_llava_image_span_from_inputs(
        model=model,
        processor=processor,
        input_ids=input_ids,
        pixel_values=pixel_values,
    )

    generated_ids = input_ids.clone()
    prompt_len = input_ids.shape[1]

    eos_token_id = model.generation_config.eos_token_id

    if eos_token_id is None:
        eos_token_ids = []
    elif isinstance(eos_token_id, int):
        eos_token_ids = [eos_token_id]
    else:
        eos_token_ids = list(eos_token_id)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        use_cache=True,
        output_attentions=True,
        return_dict=True,
    )

    if outputs.attentions is None:
        raise ValueError(
            "outputs.attentions is None. Reload model with attn_implementation='eager'."
        )

    past_key_values = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]

    for step in range(max_new_tokens):
        log_probs = F.log_softmax(next_logits, dim=-1)

        k = min(num_attn_candidates, log_probs.shape[-1])
        top_scores, top_tokens = torch.topk(log_probs, k=k, dim=-1)

        candidate_final_scores = []
        candidate_tokens = []
        candidate_outputs = []

        for cand_idx in range(k):
            cand_token = top_tokens[:, cand_idx:cand_idx + 1]
            cand_log_score = top_scores[:, cand_idx]

            cand_attention_mask = torch.cat(
                [
                    attention_mask,
                    torch.ones(
                        (attention_mask.shape[0], 1),
                        dtype=attention_mask.dtype,
                        device=attention_mask.device,
                    ),
                ],
                dim=-1,
            )

            cand_outputs = model(
                input_ids=cand_token,
                attention_mask=cand_attention_mask,
                past_key_values=past_key_values,
                use_cache=True,
                output_attentions=True,
                return_dict=True,
            )

            penalty = compute_opera_overtrust_penalty(
                attentions=cand_outputs.attentions,
                key_position=key_position,
                scale_factor=scale_factor,
                threshold=threshold,
                penalty_weights=penalty_weights,
            )

            adjusted_score = cand_log_score - penalty

            candidate_final_scores.append(adjusted_score)
            candidate_tokens.append(cand_token)
            candidate_outputs.append(cand_outputs)

        candidate_final_scores = torch.stack(candidate_final_scores, dim=1)
        best_idx = int(torch.argmax(candidate_final_scores, dim=-1).item())

        next_token = candidate_tokens[best_idx]
        selected_outputs = candidate_outputs[best_idx]

        generated_ids = torch.cat([generated_ids, next_token], dim=-1)

        attention_mask = torch.cat(
            [
                attention_mask,
                torch.ones(
                    (attention_mask.shape[0], 1),
                    dtype=attention_mask.dtype,
                    device=attention_mask.device,
                ),
            ],
            dim=-1,
        )

        past_key_values = selected_outputs.past_key_values
        next_logits = selected_outputs.logits[:, -1, :]

        if len(eos_token_ids) > 0 and int(next_token.item()) in eos_token_ids:
            break

    new_tokens = generated_ids[:, prompt_len:]

    answer = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0]

    return clean_output(answer)

In [ ]:
def opera_generate_batch_prompts_hf(
    model,
    processor,
    images: List[Image.Image],
    prompts: List[str],
    max_new_tokens: int = 256,
    num_attn_candidates: int = 5,
    scale_factor: float = 50.0,
    threshold: float = 15.0,
    penalty_weights: float = 1.0,
):
    assert len(images) == len(prompts), "images and prompts must have same length."

    outputs = []

    # Internally sequential because each sample has candidate KV-cache probes.
    for image, prompt in zip(images, prompts):
        output = opera_generate_one_hf(
            model=model,
            processor=processor,
            image=image,
            prompt=prompt,
            max_new_tokens=max_new_tokens,
            num_attn_candidates=num_attn_candidates,
            scale_factor=scale_factor,
            threshold=threshold,
            penalty_weights=penalty_weights,
        )

        outputs.append(output)

    return outputs

In [ ]:
with open(QUERY_GENERATIVE_PATH, "r", encoding="utf-8") as f:
    queries = json.load(f)

queries = [
    q for q in queries
    if MIN_ID <= int(q["id"]) <= MAX_ID
]

queries = sorted(queries, key=lambda x: int(x["id"]))

print(f"Loaded {len(queries)} AMBER generative samples.")
print("Example:", queries[0])

In [ ]:
model.eval()

output_path = Path(OUTPUT_PATH)
amber_img_path = Path(AMBER_IMG_PATH)

results = []
done_ids = set()

if output_path.exists():
    with open(output_path, "r", encoding="utf-8") as f:
        results = json.load(f)

    done_ids = set(int(row["id"]) for row in results)

    print(f"Resume mode: loaded {len(results)} existing responses.")

remaining_queries = [
    q for q in queries
    if int(q["id"]) not in done_ids
]

print(f"Remaining samples: {len(remaining_queries)}")

batches = list(batch_list(remaining_queries, BATCH_SIZE))

for batch in tqdm(batches, desc="AMBER OPERA inference"):
    images = []
    prompts = []
    ids = []

    for item in batch:
        image_id = int(item["id"])
        image_name = item["image"]
        user_query = item["query"]

        image_path = amber_img_path / image_name

        if not image_path.exists():
            matches = list(amber_img_path.rglob(image_name))

            if len(matches) == 0:
                raise FileNotFoundError(
                    f"Cannot find {image_name} under {amber_img_path}"
                )

            image_path = matches[0]

        image = Image.open(image_path).convert("RGB")

        hf_prompt = f"USER: <image>\n{user_query}\nASSISTANT:"

        images.append(image)
        prompts.append(hf_prompt)
        ids.append(image_id)

    responses = opera_generate_batch_prompts_hf(
        model=model,
        processor=processor,
        images=images,
        prompts=prompts,
        max_new_tokens=MAX_NEW_TOKENS,
        num_attn_candidates=NUM_ATTN_CANDIDATES,
        scale_factor=SCALE_FACTOR,
        threshold=THRESHOLD,
        penalty_weights=PENALTY_WEIGHTS,
    )

    for image_id, response in zip(ids, responses):
        results.append({
            "id": int(image_id),
            "response": clean_output(response),
        })

    results = sorted(results, key=lambda x: int(x["id"]))
    save_json_atomic(output_path, results)

    del images
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nSaved {len(results)} responses to {OUTPUT_PATH}")